In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import os
import matplotlib.cm as cm
from matplotlib.colors import Normalize

In [ ]:
def plot_heatmap(data_dir, info_dir, n_bins=50):
    info = pd.read_csv(info_dir)
    info['cos_theta'] = [math.cos(math.radians(theta)) for theta in info['theta']]
    info['cos_phi'] = [math.cos(math.radians(phi)) for phi in info['phi']]
    bins = np.linspace(-1, 1, n_bins+1)
    info['cos_theta_bins'] = pd.cut(info['cos_theta'],  bins, labels=False)
    info['cos_phi_bins'] = pd.cut(info['cos_phi'], bins, labels=False)
    
    xs = np.load(os.path.join(data_dir, os.listdir(data_dir)[0]))[0] # might change later
    nans = np.full(xs.shape, np.nan)
    theta_map = []
    phi_map = []

    for i in range(n_bins):
        thetas = info[info['cos_theta_bins'] == i]['source']
        phis = info[info['cos_phi_bins'] == i]['source']
        if len(thetas) != 0:
            spectra = np.array([np.load(os.path.join(data_dir, file))[1] for file in os.listdir(data_dir) if file[:-4] in thetas])
            avg = np.mean(spectra, axis=0)
            theta_map.append(avg)
        else:
            theta_map.append(nans)
        if len(phis) != 0:
            spectra = np.array([np.load(os.path.join(data_dir, file))[1] for file in os.listdir(data_dir) if file[:-4] in phis])
            avg = np.mean(spectra, axis=0)
            phi_map.append(avg)
        else:
            phi_map.append(nans)
    
    fig, axs = plt.subplots(2, 1)
    plt.subplots_adjust(hspace=0.01)

    norm = Normalize(vmin=1.00, vmax=1.05)
    im0 = axs[0].imshow(theta_map, origin='lower', norm=norm)
    im1 = axs[1].imshow(phi_map, origin='lower', norm=norm)
    axs[0].set_ylabel('$\mathrm{cos}({\\theta})$')
    axs[1].set_ylabel('$\mathrm{cos}({\phi})$')
    axs[1].set_xlabel('Frequency (MHz)')
    for ax in axs:
        freq_diff = xs[1] - xs[0]
        # Set ticks "intelligently"
        
        labels = np.arange(8505, 8535, 5)
        ax.set_xticks((labels - min(xs)) * (1 / freq_diff))
        ax.set_yticks([-0.5,24.5,49.5])
        ax.set_yticklabels([-1,0,1])
        ax.set_facecolor(cm.get_cmap('viridis')(0))
        # lines
        # for i in np.arange(0, 50, 5):
        #     ax.axhline(i, color='red')
    tick_locs = ax[0].get_xticks
    axs[0].set_xticklabels([])
    axs[1].set_xticklabels([round(loc * freqs + min(xs)) for loc in tick_locs])
    cbar = fig.colorbar(im0, ax=axs.ravel().tolist(), orientation='vertical', fraction=0.0335, pad=0.04, ticks=[1.00, 1.0125, 1.025, 1.0375, 1.05])
    cbar.set_label('Normalized Power')